# PDF Document Inspection

This notebook checks document size, page count, and text extraction quality before moving on to chunking and retrieval.

In [ ]:
from pathlib import Path

import re

import fitz  # PyMuPDF

In [ ]:
pdf_candidates = [
    Path("RAG Without Frameworks/knowledge-base/AWS Certified Cloud Practitioner Slides.pdf"),
    Path("knowledge-base/AWS Certified Cloud Practitioner Slides.pdf"),
]

for candidate in pdf_candidates:
    if candidate.is_file():
        pdf_path = candidate
        break
else:
    raise FileNotFoundError("Could not find the PDF in the expected knowledge-base folders.")

print("Using PDF:", pdf_path)

In [ ]:
def extract_pages(path: str, start_page: int = 0) -> list[dict[str, object]]:
    """Extract page text with page metadata preserved."""
    with fitz.open(path) as pdf:
        documents = []
        for page_num, page in enumerate(pdf[start_page:], start=start_page + 1):
            documents.append({
                "page": page_num,
                "text": page.get_text(),
            })
    return documents


skip_pages = 9
# The first 9 pages are speaker-introduction pages, so we skip them before extraction.
with fitz.open(str(pdf_path)) as pdf:
    print("Pages:", len(pdf))
    print("Skipping first pages:", skip_pages)

documents = extract_pages(str(pdf_path), start_page=skip_pages)
text = "\n".join(document["text"] for document in documents)
print("Pages extracted:", len(documents))
print("First extracted page:", documents[0]["page"] if documents else None)
print("Last extracted page:", documents[-1]["page"] if documents else None)
print(f"Characters after skipping first {skip_pages} pages: {len(text)}")

# Preserve Page Metadata

Instead of collapsing the PDF into a single string immediately, the notebook now keeps a per-page record in `documents`.

Each item stores:

- the source page number
- the extracted text for that page

This is useful later because any chunk can be traced back to something like `Source: Page 143` during retrieval or answer citation.

# Clean Repeated Boilerplate

The extracted PDF text still includes repeated footer and copyright strings on many pages. Those phrases do not help retrieval, so we remove them before sampling or chunking.

This step keeps the notebook focused on useful content and reduces noise in the final embeddings.

We also normalize whitespace and strip URLs so the cleaned text reads like plain content instead of layout artifacts.

In [ ]:
import re

# These phrases repeat across the slides and act like boilerplate noise in RAG.
remove_phrases = [
    "\u00a9 Stephane Maarek",
    "NOT FOR DISTRIBUTION",
    "www.datacumulus.com",
]


def clean_text(text: str) -> str:
    """Remove repeated boilerplate, URLs, and excess whitespace."""
    for phrase in remove_phrases:
        text = text.replace(phrase, "")

    text = re.sub(r"https?://\\S+", "", text)
    text = re.sub(r"\\n+", "\\n", text)
    text = re.sub(r"\\s+", " ", text)

    lines = []
    for line in text.splitlines():
        line = line.strip()
        if line:
            lines.append(line)

    return "\n".join(lines)


# Count the repeated phrases first so the cleaning step is easy to verify.
for phrase in remove_phrases:
    print(f"{phrase}: {text.count(phrase)}")

cleaned_text = clean_text(text)
print("Original characters:", len(text))
print("Cleaned characters:", len(cleaned_text))
print("\n--- Cleaned Sample ---")
print(cleaned_text[:1000])

# Check Cleaned Text Quality

Now inspect the cleaned text, not the raw extraction. This confirms that the repeated boilerplate was removed and that the remaining content is readable enough for chunking.

If the samples still look noisy, the cleaning rules should be expanded before moving on.

In [ ]:
text_to_inspect = cleaned_text

print(text_to_inspect[:1000])

print("\n--- Sample: 10000:11000 ---")
print(text_to_inspect[10000:11000])

print("\n--- Sample: 50000:51000 ---")
print(text_to_inspect[50000:51000])

print("\n--- Page Metadata Preview ---")
print([
    {
        "page": document["page"],
        "text_chars": len(document["text"]),
        "text_preview": document["text"][:120],
    }
    for document in documents[:3]
])

# Understand the RAG Flow

PDF -> Text -> Chunks -> Embeddings -> Vectors -> Similarity Search

Chunking solves the problem of large documents by splitting them into smaller, retrievable pieces that fit within model limits and support efficient retrieval-augmented generation.

At this point in the notebook, the text has already been extracted, cleaned, and validated. The next practical step would be to split it into chunks and build retrieval-ready embeddings.

Example flow:

- PDF
- Text
- Chunks
- Embeddings
- Vectors
- Similarity Search
- Best Chunks for the question

In [ ]:
# Reserved for the next step: chunking the cleaned text into retrieval-friendly pieces.
# Add chunking logic here once you are ready to move from inspection to indexing.
# Because page metadata is preserved in `documents`, you can later map each chunk back to its source page.